<a href="https://colab.research.google.com/github/SanjayGanapathy/AETHER/blob/main/Spatial_Data_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **AETHER**: an **A**nomaly **E**nsemble Framework for **T**rajectory **H**euristics & **E**xplainable **R**easoning

Migratory Ecology of the Common Kestrel (*Falco tinnunculus*)  
### Data‑Quality Assessment & Spatial Analysis  
*Author: Sanjay Ganapathy*


In [5]:
import pandas as pd

df = pd.read_csv("data.csv")
display(df.head())
display(df.info())

/tmp/ipython-input-5-251111154.py:3: DtypeWarning: Columns (15,18,22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data.csv")


,event-id,visible,timestamp,location-long,location-lat,gps:activity-count,external-temperature,gps:dop,gps:hdop,gps:satellite-count,...,sensor-type,individual-taxon-canonical-name,tag-local-identifier,individual-local-identifier,study-name,utm-easting,utm-northing,utm-zone,study-timezone,study-local-timestamp
0,29615547747,True,2023-04-28 10:06:54.000,-6.470255,37.427571,NaN,0.0,NaN,NaN,NaN,...,gps,Falco tinnunculus,1300000709,MX9208,(EBD) Common Kestrel (Falco tinnunculus) Spain...,723839.343781,4.145310e+06,29N,GMT+00:00,2023-04-28 10:06:54.000
1,29615547748,True,2023-04-28 10:26:55.000,-6.477722,37.429683,NaN,0.0,NaN,NaN,NaN,...,gps,Falco tinnunculus,1300000709,MX9208,(EBD) Common Kestrel (Falco tinnunculus) Spain...,723172.271891,4.145527e+06,29N,GMT+00:00,2023-04-28 10:26:55.000
2,29615547749,True,2023-04-28 10:28:55.000,-6.478029,37.429824,NaN,0.0,NaN,NaN,NaN,...,gps,Falco tinnunculus,1300000709,MX9208,(EBD) Common Kestrel (Falco tinnunculus) Spain...,723144.667806,4.145542e+06,29N,GMT+00:00,2023-04-28 10:28:55.000
3,29615547750,True,2023-04-28 10:31:10.000,-6.477894,37.429808,NaN,0.0,NaN,NaN,NaN,...,gps,Falco tinnunculus,1300000709,MX9208,(EBD) Common Kestrel (Falco tinnunculus) Spain...,723156.609110,4.145540e+06,29N,GMT+00:00,2023-04-28 10:31:10.000
4,29615547751,True,2023-04-28 10:32:54.000,-6.477960,37.429709,NaN,0.0,NaN,NaN,NaN,...,gps,Falco tinnunculus,1300000709,MX9208,(EBD) Common Kestrel (Falco tinnunculus) Spain...,723151.098516,4.145529e+06,29N,GMT+00:00,2023-04-28 10:32:54.000


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2534898 entries, 0 to 2534897
Data columns (total 30 columns):
 #   Column                           Dtype  
---  ------                           -----  
 0   event-id                         int64  
 1   visible                          bool   
 2   timestamp                        object 
 3   location-long                    float64
 4   location-lat                     float64
 5   gps:activity-count               float64
 6   external-temperature             float64
 7   gps:dop                          float64
 8   gps:hdop                         float64
 9   gps:satellite-count              float64
 10  gps-time-to-fix                  float64
 11  ground-speed                     float64
 12  heading                          float64
 13  height-above-msl                 float64
 14  height-raw                       float64
 15  import-marked-outlier            object 
 16  gls:light-level                  float64
 17  location

None

In [6]:
required_columns = ['location-lat', 'location-long']
if all(col in df.columns for col in required_columns):
    print(f"Columns '{required_columns[0]}' and '{required_columns[1]}' exist in the DataFrame.")
    print(f"Initial number of rows: {len(df)}")

    df.dropna(subset=required_columns, inplace=True)
    print(f"Number of rows after dropping missing lat/long: {len(df)}")

    if 'timestamp' in df.columns:
        df.dropna(subset=['timestamp'], inplace=True)
        print(f"Number of rows after dropping missing timestamp: {len(df)}")
    else:
        print("Warning: 'timestamp' column not found.")

    display(df.head())

else:
    print(f"Error: Required columns '{required_columns[0]}' and/or '{required_columns[1]}' not found in the DataFrame.")

Columns 'location-lat' and 'location-long' exist in the DataFrame.
Initial number of rows: 2534898
Number of rows after dropping missing lat/long: 2534898
Number of rows after dropping missing timestamp: 2534898


,event-id,visible,timestamp,location-long,location-lat,gps:activity-count,external-temperature,gps:dop,gps:hdop,gps:satellite-count,...,sensor-type,individual-taxon-canonical-name,tag-local-identifier,individual-local-identifier,study-name,utm-easting,utm-northing,utm-zone,study-timezone,study-local-timestamp
0,29615547747,True,2023-04-28 10:06:54.000,-6.470255,37.427571,NaN,0.0,NaN,NaN,NaN,...,gps,Falco tinnunculus,1300000709,MX9208,(EBD) Common Kestrel (Falco tinnunculus) Spain...,723839.343781,4.145310e+06,29N,GMT+00:00,2023-04-28 10:06:54.000
1,29615547748,True,2023-04-28 10:26:55.000,-6.477722,37.429683,NaN,0.0,NaN,NaN,NaN,...,gps,Falco tinnunculus,1300000709,MX9208,(EBD) Common Kestrel (Falco tinnunculus) Spain...,723172.271891,4.145527e+06,29N,GMT+00:00,2023-04-28 10:26:55.000
2,29615547749,True,2023-04-28 10:28:55.000,-6.478029,37.429824,NaN,0.0,NaN,NaN,NaN,...,gps,Falco tinnunculus,1300000709,MX9208,(EBD) Common Kestrel (Falco tinnunculus) Spain...,723144.667806,4.145542e+06,29N,GMT+00:00,2023-04-28 10:28:55.000
3,29615547750,True,2023-04-28 10:31:10.000,-6.477894,37.429808,NaN,0.0,NaN,NaN,NaN,...,gps,Falco tinnunculus,1300000709,MX9208,(EBD) Common Kestrel (Falco tinnunculus) Spain...,723156.609110,4.145540e+06,29N,GMT+00:00,2023-04-28 10:31:10.000
4,29615547751,True,2023-04-28 10:32:54.000,-6.477960,37.429709,NaN,0.0,NaN,NaN,NaN,...,gps,Falco tinnunculus,1300000709,MX9208,(EBD) Common Kestrel (Falco tinnunculus) Spain...,723151.098516,4.145529e+06,29N,GMT+00:00,2023-04-28 10:32:54.000


In [18]:
import folium
import os

# Take a random sample of the data
# Reduced sample size and changed random_state
sample_df = df.sample(n=50000, random_state=55)

mean_lat = sample_df['location-lat'].mean()
mean_lon = sample_df['location-long'].mean()

print("Creating map object...")
m = folium.Map(location=[mean_lat, mean_lon], zoom_start=5)
print("Map object created.")

print("Adding markers to the map...")
for index, row in sample_df.iterrows():
    # Changed from folium.Marker to folium.CircleMarker
    folium.CircleMarker(
        location=[row['location-lat'], row['location-long']],
        radius=2,  # Adjust the radius as needed
        color='blue', # Adjust the color as needed
        fill=True,
        fill_color='blue' # Adjust the fill color as needed
    ).add_to(m)
print("Markers added.")

# Using a different file path
file_path = 'kestrel_migration_map_sample_2.html'
try:
    print(f"Attempting to save map to {file_path}...")
    m.save(file_path)
    print(f"Map saved successfully to {file_path}.")

    # Verify file size
    if os.path.exists(file_path):
        file_size = os.path.getsize(file_path)
        print(f"Size of saved file: {file_size} bytes.")
        if file_size == 0:
            print("Warning: Saved file is empty (0 bytes).")
    else:
        print(f"Error: File {file_path} was not found after saving.")

except Exception as e:
    print(f"An error occurred while saving the map: {e}")


print(f"Number of data points in the sample: {len(sample_df)}")

Creating map object...
Map object created.
Adding markers to the map...
Markers added.
Attempting to save map to kestrel_migration_map_sample_2.html...
Map saved successfully to kestrel_migration_map_sample_2.html.
Size of saved file: 23742085 bytes.
Number of data points in the sample: 50000


In [16]:
print(f"Total number of data points in the dataset: {len(df)}")

Total number of data points in the dataset: 2534898


In [19]:
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

Number of rows: 2534898
Number of columns: 30
